# Adaptive-Compute Transformer — Adaptive-Depth Training (Colab)

Trains the adaptive-depth model (`AdaptiveTransformer` + `LearnedGateRouter`) from `adaptive-compute-transformer/src`, alongside the already-completed baseline (see `colab_train_baseline.ipynb`).
Built module by module: Setup → Code → Data → W&B → Train → Results.
Runtime > Change runtime type > GPU (T4) before running.

## Module 1 — Setup

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

!pip install -q tiktoken datasets wandb

import random, numpy as np
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("seed set:", SEED)

## Module 2 — Code

Materializes `src/` from the repo (config.py, router.py, model.py, losses.py, data.py, train_adaptive.py) into the Colab filesystem, verbatim, one `%%writefile` cell per file.

In [ ]:
%%writefile config.py
"""Config for the baseline static-depth transformer.

Every training run resolves one of these to a dict and dumps it to
experiments/<run_name>/config.json before training starts, so no result
is ever reported without the exact config + seed that produced it.
"""
from dataclasses import dataclass, asdict
from typing import Optional


@dataclass
class BaselineConfig:
    # architecture
    vocab_size: int = 50257  # gpt2 tiktoken encoding
    block_size: int = 256    # context length
    n_layer: int = 12
    n_head: int = 6
    d_model: int = 384
    d_ff: int = 1536         # 4 * d_model
    dropout: float = 0.1
    bias: bool = True

    # data
    dataset_name: str = "Salesforce/wikitext"
    dataset_config: str = "wikitext-103-raw-v1"

    # optimization
    batch_size: int = 32
    grad_accum_steps: int = 1
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    warmup_steps: int = 500
    max_steps: int = 20000
    eval_interval: int = 500
    eval_iters: int = 100

    # run control
    seed: int = 1337
    run_name: str = "baseline_v1"
    out_dir: str = "experiments"
    device: str = "cuda"
    dtype: str = "bfloat16"  # falls back to float32 in train.py if unsupported
    compile: bool = True
    wandb_project: str = "adaptive-compute-transformer"
    wandb_mode: str = "offline"  # "online" once an API key is configured

    def to_dict(self):
        return asdict(self)


@dataclass
class AdaptiveConfig(BaselineConfig):
    """Same architecture as BaselineConfig (identical data + compute
    budget, per the project's baseline-vs-adaptive comparison) plus the
    router's own hyperparameters and the compute-budget loss (see
    losses.py). Phase 1 trains with a soft gate — the LM loss alone never
    rewards using fewer layers, so target_avg_layers/budget_weight supply
    the missing pressure. Train separate runs at different
    target_avg_layers values to trace the accuracy-vs-avg-layers curve."""

    # router
    router_type: str = "learned_gate"  # see router.build_router; other types added per ablation module
    router_hidden_dim: Optional[int] = None  # None = single Linear(d_model, 1), the lightest option
    topk_capacity: float = 0.5  # fixed_topk only: exact fraction of tokens selected per sequence per layer
    confidence_threshold: float = 0.4309  # confidence_threshold only: empirically calibrated via
                                           # calibrate_threshold.py against the real architecture on
                                           # real WikiText-103 validation data (untrained model) --
                                           # targets nonzero_frac~=0.5 i.e. avg_depth~=6, matching the
                                           # other two ablation arms. Re-calibrate if real training
                                           # behavior drifts meaningfully from this starting estimate.

    # compute budget
    target_avg_layers: float = 6.0  # midpoint of n_layer=12; sweep this across separate runs for the curve
    budget_weight: float = 0.01     # needs empirical tuning on real data; keep LM loss dominant early on

    # run control
    run_name: str = "adaptive_v1"
    ckpt_interval: Optional[int] = None  # None = checkpoint every eval_interval steps; set independently
                                          # to decouple checkpoint safety from (expensive) eval cadence
                                          # on long runs, so a mid-run Colab/Kaggle disconnect never
                                          # loses more than ckpt_interval steps of progress


In [ ]:
%%writefile router.py
"""Per-token routers for the adaptive-depth model.

Each router scores every token at every layer: has this token's
representation stabilized, or does it still need more depth? All router
variants (learned gate, fixed top-k, confidence threshold) share one
forward(x, block_out) call signature, so AdaptiveBlock never needs to
branch on which router type is plugged in -- routers that don't need
block_out (learned gate, fixed top-k) simply ignore it; only
ConfidenceThresholdRouter uses it.
"""
import torch
import torch.nn as nn


class LearnedGateRouter(nn.Module):
    """Outputs a per-token gate in (0, 1): ~1 means "this token still needs
    this layer", ~0 means "treat as converged, freeze it here". Multiplying
    a block's output by this gate (the soft-training scheme) keeps the
    router differentiable end-to-end via ordinary backprop through the
    residual stream — no REINFORCE / straight-through estimator needed.
    """

    def __init__(self, d_model, hidden_dim=None, bias=True):
        super().__init__()
        if hidden_dim is None:
            self.net = nn.Linear(d_model, 1, bias=bias)
        else:
            self.net = nn.Sequential(
                nn.Linear(d_model, hidden_dim, bias=bias),
                nn.GELU(),
                nn.Linear(hidden_dim, 1, bias=bias),
            )

        self.reset_gate_init()

    def reset_gate_init(self):
        """Zero-inits the last layer so every router starts at the same
        bias-controlled gate value regardless of the (random) hidden
        state it's fed, then biases it toward "continue" (~0.88). At step
        0 the base transformer hasn't learned anything yet, so router
        decisions would otherwise be arbitrary; starting near "always
        continue" avoids randomly freezing tokens before there's any
        signal worth routing on, and lets the router earn its way toward
        skipping tokens as the base representations become meaningful.

        Callable again after the fact — a wrapping model's blanket
        weight-init pass (e.g. `model.apply(_init_weights)`) will
        re-init every nn.Linear it finds, including this one, so it
        needs to be reasserted afterward.
        """
        last_linear = self.net if isinstance(self.net, nn.Linear) else self.net[-1]
        nn.init.zeros_(last_linear.weight)
        if last_linear.bias is not None:
            nn.init.constant_(last_linear.bias, 2.0)  # sigmoid(2.0) ~= 0.88

    def forward(self, x, block_out=None):
        """x: (B, T, d_model) -> gate: (B, T), values in (0, 1). block_out
        is accepted (and ignored) so every router shares one call signature
        in AdaptiveBlock.forward() -- only ConfidenceThresholdRouter
        actually needs the block's output."""
        return torch.sigmoid(self.net(x).squeeze(-1))


class FixedTopKRouter(nn.Module):
    """Mixture-of-Depths-style router: a small learned per-token scorer
    ranks all tokens in each sequence, and exactly the top `capacity`
    fraction (rounded, per sequence) get processed -- their gate is their
    own sigmoid score; everyone else gets gate = 0 exactly. Unlike
    LearnedGateRouter, average depth doesn't need a budget loss at all --
    capacity directly and exactly determines what fraction of tokens are
    selected, by construction, every forward pass.

    Note the selected tokens' gate is their SCORE, not 1 -- selection
    (which tokens rank in the top capacity fraction) isn't differentiable,
    so scaling by the score is what gives the scorer's weights a gradient
    signal at all (only from tokens that got selected in a given batch).
    This means the "capacity fraction selected" (exact, a clean
    FLOPs-comparable number) and "avg_depth" as logged elsewhere in this
    project (gates summed across layers, which for this router is
    magnitude-weighted by the score, not a plain selection count) are two
    different quantities -- both worth logging, not interchangeable.
    """

    def __init__(self, d_model, capacity, hidden_dim=None, bias=True):
        super().__init__()
        assert 0.0 < capacity <= 1.0, f"capacity must be in (0, 1], got {capacity}"
        self.capacity = capacity
        if hidden_dim is None:
            self.net = nn.Linear(d_model, 1, bias=bias)
        else:
            self.net = nn.Sequential(
                nn.Linear(d_model, hidden_dim, bias=bias),
                nn.GELU(),
                nn.Linear(hidden_dim, 1, bias=bias),
            )

    def forward(self, x, block_out=None):
        """x: (B, T, d_model) -> gate: (B, T). Exactly
        max(1, min(T, round(capacity * T))) positions per sequence are
        non-zero (= their own sigmoid score); the rest are exactly 0.
        block_out is accepted (and ignored) for signature parity with
        ConfidenceThresholdRouter -- see LearnedGateRouter.forward."""
        B, T, _ = x.shape
        scores = torch.sigmoid(self.net(x).squeeze(-1))  # (B, T)

        k = max(1, min(T, round(self.capacity * T)))
        topk_idx = scores.topk(k, dim=1).indices  # (B, k)
        mask = torch.zeros_like(scores, dtype=torch.bool)
        mask.scatter_(1, topk_idx, True)

        return scores * mask


class ConfidenceThresholdRouter(nn.Module):
    """CALM-inspired confidence-threshold router, using a stabilization
    proxy rather than the literature's full early-exit vocab-head
    approach: gates on how much a block's own output actually changes a
    token's representation, relative to the token's own current
    magnitude -- ||block(x) - x|| / ||x||. A token whose representation
    barely moves under further processing has effectively converged.

    Relative, not absolute: the residual stream's magnitude grows across
    layers in a pre-LN transformer (nothing re-normalizes it between
    blocks), so a fixed absolute delta-norm threshold would be
    miscalibrated by depth alone -- early layers rarely tripping it, late
    layers almost always tripping it, regardless of token difficulty.
    Dividing by ||x|| makes the threshold comparable at every layer.

    No learned parameters -- nothing here is trained by gradient descent
    (there's no budget loss for this router either); average depth is
    controlled purely by the fixed threshold, calibrated empirically.
    The gate is a hard 0/1 (not continuous like the other two routers):
    there's no gradient-training reason to keep it soft, since there's no
    router weight that soft-ness would need to be differentiable for --
    gradient to the underlying block's own weights (attention/MLP) still
    flows normally through the constant gate multiplier either way.
    """

    def __init__(self, threshold, eps=1e-6):
        super().__init__()
        self.threshold = threshold
        self.eps = eps

    def forward(self, x, block_out):
        """x, block_out: (B, T, d_model) -> gate: (B, T), values in
        {0.0, 1.0} exactly."""
        delta_norm = (block_out - x).norm(dim=-1)
        relative_delta = delta_norm / (x.norm(dim=-1) + self.eps)
        return (relative_delta > self.threshold).float()


def build_router(cfg):
    """Constructs the router named by cfg.router_type. Each router class
    keeps its own primitive-argument constructor (decoupled, independently
    testable in isolation) -- this factory is the one place that knows how
    to pull the right fields off cfg for each type, so adding a new router
    type (fixed top-k, confidence threshold, ...) never requires touching
    AdaptiveBlock/AdaptiveTransformer.
    """
    if cfg.router_type == "learned_gate":
        return LearnedGateRouter(cfg.d_model, hidden_dim=cfg.router_hidden_dim)
    if cfg.router_type == "fixed_topk":
        return FixedTopKRouter(cfg.d_model, capacity=cfg.topk_capacity, hidden_dim=cfg.router_hidden_dim)
    if cfg.router_type == "confidence_threshold":
        return ConfidenceThresholdRouter(threshold=cfg.confidence_threshold)
    raise ValueError(f"unknown router_type: {cfg.router_type!r}")


In [ ]:
%%writefile model.py
"""Baseline static-depth decoder-only transformer.

Standard GPT-style architecture (pre-LayerNorm, causal self-attention),
deliberately unremarkable: this is the control model. Later, the
adaptive-depth model will subclass / wrap this same block structure so
that any measured difference is attributable to routing, not to the
base architecture.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from router import build_router


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.d_model = cfg.d_model
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=cfg.bias)
        self.attn_dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.d_model, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.attn_dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
        self.act = nn.GELU()

    def forward(self, x):
        return self.dropout(self.proj(self.act(self.fc(x))))


class Block(nn.Module):
    """One transformer block. Kept as a standalone module (rather than
    inlined in the stack) because the adaptive-depth model will need to
    call blocks individually, per-token, between router decisions."""

    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class BaselineTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # weight tying
        self.tok_emb.weight = self.head.weight

        self.apply(self._init_weights)
        # scaled init for residual projections, per GPT-2 paper
        for name, p in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, (
            f"sequence length {T} exceeds block_size {self.cfg.block_size}"
        )
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
        return logits, loss

    def num_params(self, exclude_embeddings=True):
        n = sum(p.numel() for p in self.parameters())
        if exclude_embeddings:
            n -= self.pos_emb.weight.numel()
        return n

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


class AdaptiveBlock(nn.Module):
    """One transformer block wrapped with a swappable per-token router
    (see router.build_router). The router produces a gate in (0, 1) from
    the block's input and/or output (which inputs it actually needs
    depends on the router type -- see router.py); the block's contribution
    to the residual stream is blended in by that gate: gate ~= 1 applies
    the block in full, gate ~= 0 leaves the token's state unchanged (full
    attention+MLP skip, not just MLP).

    Phase 1 (soft/dense training): every token still runs through the
    full block every layer — nothing is actually skipped compute-wise
    yet, only down-weighted or masked — regardless of router type. Real
    hard-skip / token-compaction for actual compute savings is a
    separate, later phase.
    """

    def __init__(self, cfg):
        super().__init__()
        self.block = Block(cfg)
        self.router = build_router(cfg)

    def forward(self, x):
        out = self.block(x)
        gate = self.router(x, out)  # (B, T); most routers only use x, ConfidenceThresholdRouter needs out too
        g = gate.unsqueeze(-1)
        x = g * out + (1 - g) * x
        return x, gate


class AdaptiveTransformer(nn.Module):
    """Adaptive-depth counterpart to BaselineTransformer: identical
    architecture (same embeddings, ln_f, head, weight tying, init scheme)
    so any measured difference is attributable to routing — but each
    block is an AdaptiveBlock instead of a plain Block, and forward()
    additionally returns the per-layer, per-token gates so a training
    loop can log average depth used and (later) add a compute-budget
    loss term.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([AdaptiveBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # weight tying
        self.tok_emb.weight = self.head.weight

        self.apply(self._init_weights)
        # scaled init for residual projections, per GPT-2 paper
        for name, p in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))
        # self.apply() above re-initializes every nn.Linear it finds,
        # including each router's -- reassert the router's own zero-init
        # bootstrap (see LearnedGateRouter.reset_gate_init) afterward.
        # Duck-typed: not every router type has (or needs) a bootstrap
        # reset -- e.g. a non-learned confidence-threshold router has no
        # params to reassert -- so only call it where it actually exists.
        for block in self.blocks:
            if hasattr(block.router, "reset_gate_init"):
                block.router.reset_gate_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, (
            f"sequence length {T} exceeds block_size {self.cfg.block_size}"
        )
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))

        gates = []
        for block in self.blocks:
            x, gate = block(x)
            gates.append(gate)
        gates = torch.stack(gates, dim=-1)  # (B, T, n_layer)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )

        # Sum of gates across layers approximates each token's "effective
        # depth" under the soft blend (gate ~= 1 per layer contributes
        # ~1 full layer of compute); averaged over the batch, this is the
        # single number the accuracy-vs-avg-layers curve is built from.
        avg_depth = gates.sum(dim=-1).mean()
        aux = {"gates": gates, "avg_depth": avg_depth}
        return logits, loss, aux

    def num_params(self, exclude_embeddings=True):
        n = sum(p.numel() for p in self.parameters())
        if exclude_embeddings:
            n -= self.pos_emb.weight.numel()
        return n


In [ ]:
%%writefile losses.py
"""Compute-budget loss for the adaptive-depth model.

The LM cross-entropy alone only ever rewards MORE depth (more compute is
never wrong for next-token prediction) -- nothing pushes the model toward
using fewer layers. This adds the missing pressure: a penalty on how far
the batch's average effective depth (AdaptiveTransformer's
`aux["avg_depth"]`) is from a target, so sweeping that target across
separate training runs traces out the accuracy-vs-average-layers curve
that's the project's central result.

Deliberately penalizes the batch AVERAGE depth, not each token's
individual depth: the whole premise of this project is that depth should
vary a lot per token (easy tokens exit early, hard ones go deep). A
per-token penalty would fight that directly; an average-depth penalty
only constrains the aggregate compute budget and leaves per-token routing
free to be as heterogeneous as the data calls for.
"""
import torch


def adaptive_loss(lm_loss, avg_depth, cfg):
    """Combines the LM loss with a quadratic penalty pulling the batch's
    average effective depth toward cfg.target_avg_layers.

    Returns (total_loss, budget_loss) -- the unweighted budget_loss is
    returned separately so a training loop can log both terms.
    """
    target = torch.as_tensor(cfg.target_avg_layers, dtype=avg_depth.dtype, device=avg_depth.device)
    budget_loss = (avg_depth - target) ** 2
    total_loss = lm_loss + cfg.budget_weight * budget_loss
    return total_loss, budget_loss


def compute_loss(lm_loss, avg_depth, cfg):
    """Router-type-aware total loss (see router.build_router for the
    router types this dispatches on). Only learned_gate has a budget loss
    to add: its average depth is a free-floating quantity that needs an
    explicit target pushed toward via gradient descent. fixed_topk's
    average depth is already exact by construction (via its capacity
    hyperparameter), and confidence_threshold has no learned parameters
    to train toward a target at all -- for both, total loss is just the
    LM loss; average depth is still logged (see train_adaptive.py) but
    isn't being optimized against a target.

    Returns (total_loss, budget_loss_or_None).
    """
    if cfg.router_type == "learned_gate":
        return adaptive_loss(lm_loss, avg_depth, cfg)
    return lm_loss, None


In [ ]:
%%writefile data.py
"""WikiText-103 loading + tokenization + batching.

Uses HF `datasets` only for corpus I/O (streaming/download of raw text).
Tokenization (tiktoken) and batching are hand-written — no HF Trainer,
no HF data collators.
"""
import os
import numpy as np
import tiktoken

CACHE_DIR = os.path.join(os.getcwd(), "data", "tokenized")


def _tokenize_split(split, dataset_config, enc):
    from datasets import load_dataset

    ds = load_dataset("Salesforce/wikitext", dataset_config, split=split)
    eot = enc.eot_token
    ids = []
    for row in ds:
        text = row["text"]
        if not text.strip():
            continue
        ids.extend(enc.encode_ordinary(text))
        ids.append(eot)
    return np.array(ids, dtype=np.uint16)


def prepare_dataset(dataset_config="wikitext-103-raw-v1", cache_dir=CACHE_DIR):
    """Tokenizes train/validation/test splits once and caches them as
    .bin files of uint16 token ids. Safe to call every run — skips work
    if the cache already exists."""
    os.makedirs(cache_dir, exist_ok=True)
    enc = tiktoken.get_encoding("gpt2")

    paths = {}
    for split, hf_split in [("train", "train"), ("val", "validation"), ("test", "test")]:
        path = os.path.join(cache_dir, f"{split}.bin")
        paths[split] = path
        if os.path.exists(path):
            continue
        ids = _tokenize_split(hf_split, dataset_config, enc)
        ids.tofile(path)
    return paths


def get_batch(split, cfg, cache_dir=CACHE_DIR, device="cpu"):
    """Memory-maps the cached token file and samples a random batch of
    (input, target) sequences. Re-opened per call (cheap, avoids stale
    mmap handles) — standard nanoGPT-style pattern."""
    import torch

    path = os.path.join(cache_dir, f"{split}.bin")
    data = np.memmap(path, dtype=np.uint16, mode="r")

    ix = np.random.randint(0, len(data) - cfg.block_size - 1, size=cfg.batch_size)
    x = torch.stack(
        [torch.from_numpy(data[i:i + cfg.block_size].astype(np.int64)) for i in ix]
    )
    y = torch.stack(
        [torch.from_numpy(data[i + 1:i + 1 + cfg.block_size].astype(np.int64)) for i in ix]
    )
    if device != "cpu":
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x, y


In [ ]:
%%writefile train_adaptive.py
"""Hand-written training loop for the adaptive-depth transformer.

Mirrors train.py's structure and reproducibility discipline (full config +
seed + git commit logged before training starts) but swaps in
AdaptiveTransformer and a router-type-aware loss (losses.compute_loss --
only learned_gate actually has a budget-loss term; see its docstring),
and logs the extra metrics that matter for this model regardless of
router type: avg_depth (effective layers used, magnitude-weighted for
fixed_topk -- see FixedTopKRouter's docstring), nonzero_frac (the true
fraction of (token, layer) slots the router actually applied at all --
redundant with avg_depth for learned_gate/confidence_threshold, but the
one exact, FLOPs-comparable number for fixed_topk), and the LM loss
alongside the total (LM + budget, where applicable) loss that's actually
optimized, for both train and val.

No result from this script should ever be quoted without the matching
config.json sitting next to it.
"""
import os
import json
import time
import subprocess
import math

import torch

from config import AdaptiveConfig
from model import AdaptiveTransformer
from data import prepare_dataset, get_batch
from losses import compute_loss


def get_git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=os.path.dirname(__file__)
        ).decode().strip()
    except Exception:
        return "unknown"


def get_lr(step, cfg):
    if step < cfg.warmup_steps:
        return cfg.learning_rate * (step + 1) / cfg.warmup_steps
    if step > cfg.max_steps:
        return cfg.min_lr
    decay_ratio = (step - cfg.warmup_steps) / (cfg.max_steps - cfg.warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return cfg.min_lr + coeff * (cfg.learning_rate - cfg.min_lr)


@torch.no_grad()
def estimate_loss(model, cfg, device):
    model.eval()
    out = {}
    for split in ("train", "val"):
        total_losses = torch.zeros(cfg.eval_iters)
        lm_losses = torch.zeros(cfg.eval_iters)
        avg_depths = torch.zeros(cfg.eval_iters)
        nonzero_fracs = torch.zeros(cfg.eval_iters)
        for i in range(cfg.eval_iters):
            x, y = get_batch(split, cfg, device=device)
            _, lm_loss, aux = model(x, y)
            total_loss, _ = compute_loss(lm_loss, aux["avg_depth"], cfg)
            total_losses[i] = total_loss.item()
            lm_losses[i] = lm_loss.item()
            avg_depths[i] = aux["avg_depth"].item()
            nonzero_fracs[i] = (aux["gates"] > 0).float().mean().item()
        # "train"/"val" hold the TOTAL loss (LM + budget where applicable),
        # matching the key names train.py's baseline log uses for its
        # optimized loss -- so existing plotting code that reads
        # r["train"]/r["val"] keeps working unchanged on this log too.
        # LM loss, avg_depth, and nonzero_frac are separate keys.
        out[split] = total_losses.mean().item()
        out[f"{split}_lm"] = lm_losses.mean().item()
        out[f"{split}_avg_depth"] = avg_depths.mean().item()
        out[f"{split}_nonzero_frac"] = nonzero_fracs.mean().item()
    model.train()
    return out


def train(cfg: AdaptiveConfig):
    torch.manual_seed(cfg.seed)

    device = cfg.device if torch.cuda.is_available() else "cpu"
    if device == "cpu" and cfg.device == "cuda":
        print("WARNING: CUDA requested but not available — falling back to CPU. "
              "This will be very slow; use Colab/Kaggle GPU for real runs.")

    run_dir = os.path.join(cfg.out_dir, cfg.run_name)
    os.makedirs(run_dir, exist_ok=True)

    resolved = cfg.to_dict()
    resolved["git_commit"] = get_git_commit()
    resolved["resolved_device"] = device
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(resolved, f, indent=2)

    print(f"[{cfg.run_name}] preparing dataset (cached after first run)...")
    prepare_dataset(cfg.dataset_config)

    model = AdaptiveTransformer(cfg).to(device)
    print(f"[{cfg.run_name}] model params (excl. pos emb): {model.num_params():,}")

    if cfg.compile and device == "cuda":
        model = torch.compile(model)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        betas=(cfg.beta1, cfg.beta2),
        weight_decay=cfg.weight_decay,
    )

    use_wandb = False
    try:
        import wandb
        os.environ["WANDB_MODE"] = cfg.wandb_mode
        wandb.init(project=cfg.wandb_project, name=cfg.run_name, config=resolved)
        use_wandb = True
    except Exception as e:
        print(f"W&B not initialized ({e}); continuing without it.")

    log_path = os.path.join(run_dir, "train_log.jsonl")
    log_f = open(log_path, "a")

    # Checkpoint saving is decoupled from eval_interval: eval does real
    # forward passes (eval_iters batches x 2 splits) so it's expensive,
    # while a checkpoint save is just a state_dict dump -- cheap. Forcing
    # them onto the same cadence means either wasting eval compute for
    # checkpoint safety, or losing checkpoint safety when eval_interval is
    # widened for speed on a long run. ckpt_interval defaults to
    # eval_interval (unchanged behavior) but can be set independently.
    ckpt_interval = cfg.ckpt_interval or cfg.eval_interval
    ckpt_path = os.path.join(run_dir, "ckpt.pt")

    t0 = time.time()
    for step in range(cfg.max_steps + 1):
        lr = get_lr(step, cfg)
        for g in optimizer.param_groups:
            g["lr"] = lr

        if step % cfg.eval_interval == 0:
            metrics = estimate_loss(model, cfg, device)
            elapsed = time.time() - t0
            record = {"step": step, "lr": lr, "elapsed_s": elapsed, **metrics}
            print(f"step {step}: train_loss {metrics['train']:.4f} "
                  f"val_loss {metrics['val']:.4f} "
                  f"train_avg_depth {metrics['train_avg_depth']:.2f} "
                  f"val_avg_depth {metrics['val_avg_depth']:.2f} lr {lr:.2e}")
            log_f.write(json.dumps(record) + "\n")
            log_f.flush()
            if use_wandb:
                wandb.log(record, step=step)

        # Overwrites the same ckpt.pt every time, so there's always exactly
        # one recent checkpoint sitting in the run directory -- readable
        # from Colab/Kaggle's file panel even if the session disconnects
        # mid-run and train() never returns to trigger the notebook's
        # end-of-run backup/download.
        if step % ckpt_interval == 0:
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "step": step,
                "config": resolved,
            }, ckpt_path)

        x, y = get_batch("train", cfg, device=device)
        for micro_step in range(cfg.grad_accum_steps):
            _, lm_loss, aux = model(x, y)
            total_loss, _ = compute_loss(lm_loss, aux["avg_depth"], cfg)
            total_loss = total_loss / cfg.grad_accum_steps
            total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    log_f.close()
    if use_wandb:
        wandb.finish()


if __name__ == "__main__":
    cfg = AdaptiveConfig()
    train(cfg)


In [ ]:
import sys
sys.path.insert(0, '.')
import importlib
import config, router, model, losses, data, train_adaptive
for m in (config, router, model, losses, data, train_adaptive):
    importlib.reload(m)
print('code modules loaded OK:', [m.__name__ for m in (config, router, model, losses, data, train_adaptive)])

## Module 3 — Data

Downloads WikiText-103-raw-v1 via HF `datasets` and tokenizes it with the GPT-2 BPE tokenizer (`tiktoken`), caching to `.bin` files under `data/tokenized/` — same cache format/location `data.py` uses in the baseline notebook. If you're running this in the same Colab session right after the baseline run, this cell finishes instantly (already cached); in a fresh session it takes a few minutes.

In [ ]:
from data import prepare_dataset
from config import AdaptiveConfig

cfg = AdaptiveConfig()
paths = prepare_dataset(cfg.dataset_config)

import os
for split, path in paths.items():
    size_mb = os.path.getsize(path) / 1e6
    assert size_mb > 0, f"{split} cache is empty — tokenization failed"
    print(f"{split}: {path} ({size_mb:.1f} MB)")

## Module 4 — W&B (optional)

Logs in to Weights & Biases so runs are tracked online. Skip this cell to log offline (default `wandb_mode="offline"` in `config.py` — runs still work and log locally to `experiments/<run_name>/train_log.jsonl`, just not to the W&B dashboard).

In [ ]:
import wandb
wandb.login()  # paste your W&B API key when prompted

# flip to online now that we're logged in — edit cfg in the Train module below
print("W&B login OK. Set cfg.wandb_mode = 'online' in the train cell to log to the dashboard.")

## Module 5 — Train

**ROUTER_TYPE** (set once, at the top of 5a) selects which router strategy this whole module exercises: `"learned_gate"` (the original accuracy-vs-avg-layers sweep — see `TARGET_AVG_LAYERS` in 5b), `"fixed_topk"` (the router-ablation arm, and — via `TOPK_CAPACITY` — the Path A accuracy-vs-real-speed sweep; see `DECISIONS.md`, 2026-09-09), or `"confidence_threshold"` (the other router-ablation arm, no sweep). `confidence_threshold` needs no extra hyperparameter here — its empirically-calibrated `confidence_threshold=0.4309` (see `calibrate_threshold.py`) is already `AdaptiveConfig`'s default.

**5a. Smoke run** — tiny config (200 steps, small eval) on the real GPU, using the same `ROUTER_TYPE` about to run for real, to confirm the loop actually runs end-to-end for *that* router type, loss stays finite, and `avg_depth` prints a sane number (between 0 and `n_layer`) — before committing to a long run. Still not a reported result — just a correctness check. Deliberately does not vary with `TARGET_AVG_LAYERS`/`TOPK_CAPACITY` (5b's sweep variables) — the smoke run checks the training loop is correct for the router *type*, not the specific hyperparameter value about to be used for real, matching how it's always worked for `learned_gate`'s sweep.

**5b. Full run** — the real training run for whichever `ROUTER_TYPE` was set above. Config + seed + git commit are written to `experiments/<run_name>/config.json` before training starts, per the project's reproducibility rule. Two layers of checkpoint safety: (1) `train_adaptive.py` itself now overwrites `ckpt.pt` in the run directory every `ckpt_interval` steps (defaults to `eval_interval`) *during* training — independent of `train()` returning, so a mid-run disconnect still leaves a recent checkpoint sitting in Colab/Kaggle's file panel; (2) **the checkpoint + logs are additionally zipped and downloaded automatically at the end of this same cell**, right after `train()` returns, for the normal case where the run completes cleanly.


In [ ]:
from train_adaptive import train
from config import AdaptiveConfig

# ROUTER_TYPE selects which router strategy this whole Train module
# exercises -- smoke run (5a), full run (5b), and Module 6's results
# plot below all reuse this one variable, so changing it here is the one
# place to switch which ablation arm you're about to run for real.
ROUTER_TYPE = "learned_gate"  # "learned_gate" | "fixed_topk" | "confidence_threshold"

smoke_cfg = AdaptiveConfig(
    run_name="smoke_test_adaptive_colab",
    router_type=ROUTER_TYPE,
    max_steps=200,
    eval_interval=50,
    eval_iters=10,
    warmup_steps=20,
    compile=False,  # skip torch.compile for a fast smoke run
    wandb_mode="offline",
)
train(smoke_cfg)
print("SMOKE RUN COMPLETE — check that train_loss decreased and train_avg_depth is a sane, "
      "finite number above before running the full config.")

In [ ]:
# Full adaptive-depth run -- only proceed once the smoke run above shows
# decreasing loss and a sane avg_depth.
#
# ROUTER_TYPE was set in the smoke-run cell above and is reused here so
# the real run always matches whatever the smoke run just checked.
#
# SWEEP (learned_gate): for the accuracy-vs-avg-layers curve, TARGET_AVG_LAYERS
# is the one line to change per run -- everything else (seed, budget_weight,
# architecture) stays fixed so it's the only isolated variable. Run names are
# self-documenting (adaptive_target<N>) so plot_curve.py doesn't need to parse
# anything beyond each run's own config.json.
#
# SWEEP (fixed_topk, Path A -- see DECISIONS.md 2026-09-09): TOPK_CAPACITY is
# the analogous one line to change per run, for the accuracy-vs-real-speed
# curve. 0.5 is the already-completed ablation_topk run (don't re-run it --
# RUN_NAMES maps 0.5 back to that same name); any other value gets its own
# self-documenting run name (ablation_topk_cap<N>, N = capacity as a whole-
# number percentage) so the new capacity-curve script doesn't need to parse
# anything beyond each run's own config.json either. Module 1's diagnostic
# (src/diagnose_offcapacity.py) confirmed a genuine independent training run
# is required per capacity point -- there's no cheaper inference-time shortcut.
#
# ABLATION (confidence_threshold): no extra hyperparameter to set here --
# confidence_threshold=0.4309 is already the right, calibrated AdaptiveConfig
# default (see calibrate_threshold.py).
TARGET_AVG_LAYERS = 6.0  # learned_gate only
TOPK_CAPACITY = 0.5      # fixed_topk only

RUN_NAMES = {
    "learned_gate": f"adaptive_target{TARGET_AVG_LAYERS:g}",
    "fixed_topk": "ablation_topk" if TOPK_CAPACITY == 0.5 else f"ablation_topk_cap{round(TOPK_CAPACITY * 100)}",
    "confidence_threshold": "ablation_confidence",
}

full_cfg = AdaptiveConfig(
    run_name=RUN_NAMES[ROUTER_TYPE],
    router_type=ROUTER_TYPE,
    target_avg_layers=TARGET_AVG_LAYERS,  # only used when ROUTER_TYPE == "learned_gate"
    topk_capacity=TOPK_CAPACITY,          # only used when ROUTER_TYPE == "fixed_topk"
    wandb_mode="online",  # change to "offline" if you skipped the W&B login cell
)
train(full_cfg)

# --- Checkpoint backup, in the SAME cell, immediately after training ---
# The baseline run's checkpoint was lost to a Colab runtime disconnect that
# happened before it was downloaded (Colab's /content storage is wiped on
# disconnect) -- see memory.md Session 4. Never treat a run as complete
# until the checkpoint has actually been pulled off Colab.
import os, shutil
run_dir = os.path.join(full_cfg.out_dir, full_cfg.run_name)
backup_name = f"{full_cfg.run_name}_backup"
shutil.make_archive(backup_name, "zip", run_dir)
print(f"Backed up {run_dir} -> {backup_name}.zip")

try:
    from google.colab import files
    files.download(f"{backup_name}.zip")
    print("Download triggered — save it somewhere durable (e.g. Google Drive/local disk) "
          "before doing anything else.")
except ImportError:
    print("Not running in Colab — skipping download, but the zip is at:",
          os.path.abspath(f"{backup_name}.zip"))


## Module 6 — Results

Plots train/val total loss (LM + budget) **and** the `avg_depth` trajectory against its target from the actual `train_log.jsonl` written by the run above — nothing here is estimated. Run this after Module 5b's backup has already completed, so reviewing results never blocks or risks the checkpoint backup.

In [ ]:
import json, os
import matplotlib.pyplot as plt

run_name = full_cfg.run_name
run_dir = os.path.join(full_cfg.out_dir, run_name)
log_path = os.path.join(run_dir, "train_log.jsonl")

records = [json.loads(line) for line in open(log_path)]
steps = [r["step"] for r in records]
train_loss = [r["train"] for r in records]
val_loss = [r["val"] for r in records]
train_avg_depth = [r["train_avg_depth"] for r in records]
val_avg_depth = [r["val_avg_depth"] for r in records]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7), sharex=True)

ax1.plot(steps, train_loss, label="train loss (LM + budget)")
ax1.plot(steps, val_loss, label="val loss (LM + budget)")
ax1.set_ylabel("total loss"); ax1.legend()
ax1.set_title(f"Adaptive transformer ({full_cfg.router_type}) — {run_name}")

ax2.plot(steps, train_avg_depth, label="train avg_depth")
ax2.plot(steps, val_avg_depth, label="val avg_depth")
# target_avg_layers only drives training for learned_gate -- for
# fixed_topk/confidence_threshold it is unused, so drawing it here would
# misleadingly imply a target that nothing was actually optimized against.
if full_cfg.router_type == "learned_gate":
    ax2.axhline(full_cfg.target_avg_layers, color="gray", linestyle="--",
                label=f"target ({full_cfg.target_avg_layers})")
ax2.set_xlabel("step"); ax2.set_ylabel("avg layers used"); ax2.legend()

plt.tight_layout()

results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
plt.savefig(f"{results_dir}/{run_name}_loss_and_depth.png", dpi=150)
plt.show()

target_str = f", target={full_cfg.target_avg_layers}" if full_cfg.router_type == "learned_gate" else ""
print(f"final train_loss={train_loss[-1]:.4f}  val_loss={val_loss[-1]:.4f}  "
      f"train_avg_depth={train_avg_depth[-1]:.2f}  val_avg_depth={val_avg_depth[-1]:.2f}  "
      f"(router_type={full_cfg.router_type}{target_str}, step {steps[-1]})")